# Digital Finance Scraping
Crawl all volumes and issues of Springer *Digital Finance* (journal 42521) for:  
publication year, title, abstract, Volume, Issue, and PDF page count.  
Output: digital_finance_metadata.csv  
Dependencies: pip install requests beautifulsoup4 lxml pypdf  

In [1]:
import requests, time, io, csv, random
from bs4 import BeautifulSoup
from pypdf import PdfReader

In [2]:
BASE     = "https://link.springer.com"
JOURNAL  = "42521"
CSV_FILE = "Data/digital_finance_metadata.csv"
# Volume-issue structure: {volume number: [list of issue URL numbers]}
VOLUMES = {
    8: [1, 2, 3, 4],   # 2026
    7: [1, 2, 3, 4],   # 2025
    6: [1, 2, 3, 4],   # 2024
    5: [1, 2, 3],      # 2023, Issue 3-4
    4: [1, 2, 4],      # 2022, Issue 2-3
    3: [1, 2, 3],      # 2021, Issue 3-4
    2: [1, 3],         # 2020, Issue 1-2 / 3-4
    1: [1],            # 2019, Issue 1-4
}

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/125.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})

In [3]:
def get_soup(url):
    r = session.get(url, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "lxml")


def meta(soup, name):
    tag = soup.find("meta", attrs={"name": name})
    return tag["content"].strip() if tag and tag.get("content") else None


def get_article_links(issue_url):
    """Extract relative paths of all articles from an issue page"""
    links = []
    for a in get_soup(issue_url).select("a[href*='/article/10.1007/']"):
        href = a["href"].split("?")[0]
        if href not in links:
            links.append(href)
    return links


def crawl_article(path, vol, iss):
    """Scrape metadata of a single article"""
    s = get_soup(BASE + path)
    abs_div = s.find("div", id="Abs1-content")

    pdf_url = meta(s, "citation_pdf_url")
    pdf_pages = None
    if pdf_url:
        try:
            r = session.get(pdf_url, timeout=180)
            if r.ok and r.content[:4] == b"%PDF":
                pdf_pages = len(PdfReader(io.BytesIO(r.content)).pages)
        except Exception as e:
            print(f"    PDF download error: {e}")

    return {
        "year": (meta(s, "citation_publication_date") or "")[:4],
        "title": meta(s, "citation_title") or "",
        "abstract": abs_div.get_text(" ", strip=True) if abs_div else "",
        "volume": meta(s, "citation_volume") or vol,
        "issue": meta(s, "citation_issue") or iss,
        "pdf_pages": pdf_pages,
    }


def main():
    fields = ["year", "volume", "issue","title", "abstract",  "pdf_pages"]

    with open(CSV_FILE, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()

        for vol, issues in VOLUMES.items():
            for iss in issues:
                issue_url = f"{BASE}/journal/{JOURNAL}/volumes-and-issues/{vol}-{iss}"
                print(f"\n===== Volume {vol}, Issue {iss} =====")

                for i, path in enumerate(get_article_links(issue_url), 1):
                    row = crawl_article(path, vol, iss)
                    writer.writerow(row)
                    f.flush()
                    print(f"  [{i}] {row['title'][:60]} | {row['year']} | PDF pages: {row['pdf_pages']}")
                    time.sleep(2 + random.uniform(0, 2))   # Random Sleep

    print(f"\nDone. Results saved to {CSV_FILE}")


if __name__ == "__main__":
    main()


===== Volume 8, Issue 1 =====
  [1] Liquidity provider position analysis and pricing in automate | 2026 | PDF pages: 28
  [2] Social media discussion and short-horizon stock returns: evi | 2026 | PDF pages: 17
  [3] Explaining digital payments adoption with econometrics and e | 2026 | PDF pages: 23
  [4] The self-adaptive Cuckoo genetic algorithm for echo state ne | 2026 | PDF pages: 29
  [5] Bitcoin, U.S. stock markets, and volatility: the interaction | 2026 | PDF pages: 25
  [6] Option-based pricing of secured lending in P2P crypto market | 2026 | PDF pages: 23
  [7] Adaptive bagging strategies to address concept drift and ret | 2026 | PDF pages: 30
  [8] ES-LSTM: a hybrid model for accurate time series forecasting | 2026 | PDF pages: 21
  [9] Market-specific connectedness behaviors across quantiles and | 2026 | PDF pages: 45
  [10] AlphaNiftyAI : conquering the Indian stock market with AI | 2026 | PDF pages: 28
  [11] Financial outliers in the digital cosmos: time–frequency ins | 2